# Correlation between heart rate variability & reported user immersion in an VR experience

* Lucas Friborg Mitchell - lmitch21@student.aau.dk - Study No. 20213721


## Introduction

Quantifying a qualitative experience has always been a challenge. Using physiological data can help bridge the gap.

This mini-project was chosen to support a main semester project. As such, the data-set and processing will be mostly the same. This paper will however go more into depth on how the data was processed.

The semester project goal is to see if their is a correlation between task performance and level of immersion. An immersive VR experience is created, where the player has to make and serve cocktails for costumers. The game is split up into 3 different versions of increasing visual and sensory fidelity. 

The experiment is a between-subjects design, where each participant takes a 5 minute baseline measurement, followed by 1 of the 3 fidelity conditions. They then have 8 minutes to complete as many orders as possible.

## Implementation 

The heart data was acquired, using the Blood Volume Pulse (BVP) finger clip sensor from Plux Biosignals. As such, the data was streamed from OpenSignals and recorded with [LabRecorder](https://github.com/labstreaminglayer/App-LabRecorder). This allowed for simple management of multiple data streams and synchronization.

In [1]:
from scipy.signal import butter, filtfilt, find_peaks
import neurokit2 as nk
import numpy as np
import pyxdf
import matplotlib.pyplot as plt
import pandas as pd
from biosppy.signals.ppg import ppg
import heartpy as hp
import glob
import re
import os
from scipy import stats
from scipy.stats import shapiro, levene, f_oneway, kruskal, mannwhitneyu
from scipy.stats import f_oneway
from itertools import combinations
import seaborn as sns
from factor_analyzer import FactorAnalyzer
import pingouin as pg
import os

debugging = False
def Trace(message):
    """
    Function to print messages with a specific format.
    """
    if (debugging):
        print(f"[Trace] {message}")
    else:
        pass


In [2]:
def get_gaze_data(filepath = "/home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/ses-13/sub-13_ses-13_task-Baseline/_.xdf"):
    try:
        streams, header = pyxdf.load_xdf(filepath)
    except Exception as e:
        print(f"Error loading XDF file {filepath}: {e}")
        return None, None
    gaze_stream = next((s for s in streams if s["info"]["name"][0] == "GazePointStream"), None)
    if gaze_stream:
        gaze_data = gaze_stream['time_series']
        gaze_ts = gaze_stream['time_stamps']
        return gaze_data, gaze_ts
    else:
        print(f"Warning: Gaze stream not found in {filepath}.")
        return None, None

gaze_data, gaze_ts = get_gaze_data()
# convert to DataFrame with time stamps in one axis and the object being looked at in the other axis
gaze_df = pd.DataFrame({'time': gaze_ts, 'layer': [l[0] for l in gaze_data]})
# print each unique layer
print(gaze_df["layer"].unique())

['0']


### Signal extraction
The first step is extracting the data stream from the file. The .XDF file contains dictionaries, formatted as XML. So we define a path to the specific dictionary and save the signal stream and time series to a list each.

In [3]:
def extract_bvp_and_markers_from_xdf(filepath):
    #print(f"Attempting to load XDF: {filepath}")
    try:
        streams, header = pyxdf.load_xdf(filepath)
    except Exception as e:
        print(f"Error loading XDF file {filepath}: {e}")
        return None, None, pd.DataFrame(), {'bvp': False, 'markers': False}
    bvp_s = next((s for s in streams if s["info"]["name"][0] == "OpenSignals" and any(ch['label'][0] == 'BVP0' for ch in s['info']['desc'][0]['channels'][0]['channel'])), None)
    mark_s = next((s for s in streams if s["info"]["name"][0] == "UnityMarkers"), None)
    streams_found = {'bvp': False, 'markers': False}
    bvp_signal, bvp_ts = None, None
    df_events = pd.DataFrame()
    if bvp_s:
        bvp_data_raw = bvp_s['time_series']
        bvp_ts_raw = bvp_s['time_stamps']
        bvp_channel_idx = next((i for i, ch in enumerate(bvp_s['info']['desc'][0]['channels'][0]['channel']) if ch['label'][0] == 'BVP0'), None)
        if bvp_channel_idx is not None and bvp_data_raw.ndim == 2 and bvp_data_raw.shape[1] > bvp_channel_idx:
            bvp_signal = bvp_data_raw[:, bvp_channel_idx].astype(np.float64)
            bvp_ts = bvp_ts_raw
            streams_found['bvp'] = True
            Trace(f"BVP stream found and BVP0 channel extracted from {filepath}.")
        else:
            print(f"Warning: BVP0 channel not found or data format unexpected in OpenSignals stream for {filepath}.")
    else:
        print(f"Warning: BVP stream (OpenSignals with BVP0) not found in {filepath}.")
    if mark_s:
        markers_raw = mark_s['time_series']
        marker_ts_raw = mark_s['time_stamps']
        if len(markers_raw) > 0:
            df_events = pd.DataFrame({'time': marker_ts_raw, 'event': [m[0] for m in markers_raw]})
            streams_found['markers'] = True
            #print(f"UnityMarkers stream found in {filepath}.")
        #else:
            #print(f"Warning: UnityMarkers stream found but no event data in {filepath}.")
    return bvp_signal, bvp_ts, df_events, streams_found


### Butterworth Bandpass Filter

Next, is filtering the raw data with a Butterworth filter [(Butterworth, S. (1930))](https://www.changpuak.ch/electronics/downloads/On_the_Theory_of_Filter_Amplifiers.pdf). It is commonly used for filtering ECG signals, as it helps remove high frequency noise. But it also works well for other types of signals to reduce noise.
The equation for the for a Butterworth Filter is given by:
$$
    G(w) = \frac{1}{\sqrt{1+w^2n}}
$$

Where $w$ is the angular frequency in radiants per second and $n$ is the number of poles in the filter. This can be modified into a bandpass filter to look like: 
$$
    G_{BP}(w) = \frac{1}{\sqrt{1+(\frac{B(w^2-w_0^2)}{ww_0})^{2n}}}
$$
Here $w_0 = \sqrt{w_Lw_H}$ as the low and high frequency cutoffs, and $B = w_H - w_L$ being the bandwidth.

The sample rate is based on the Nyquist frequency [(Shannon, Claude E. (1949))](https://doi.org/10.1109%2Fjrproc.1949.232969). Since we don't expect to see a heart rate of more than 200 BPM, the chosen sample rate will be 400 Hz.

In [4]:
def filter_bvp(signal, lowcut=0.5, highcut=8.0, fs=400):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    if high <= low:
        print(f"Warning: Highcut frequency ({highcut}Hz) is not above lowcut frequency ({lowcut}Hz) at fs={fs}Hz. Adjusting filter or skipping.")
        return signal
    try:
        b, a = butter(4, [low, high], btype='bandpass')
        filtered = filtfilt(b, a, signal)
        # plot before and after filtering for debugging
        if debugging:
            plt.figure(figsize=(12, 6))
            plt.subplot(2, 1, 1)
            plt.plot(signal, label='Original Signal')
            plt.title('Original Signal')
            plt.subplot(2, 1, 2)
            plt.plot(filtered, label='Filtered Signal', color='orange')
            plt.title('Filtered Signal')
            plt.tight_layout()
            plt.show()
        return filtered
    except ValueError as ve:
        print(f"ValueError during filtering: {ve}. Returning unfiltered signal.")
        return signal

### Calculating Heart Rate Variability

[HeartPy](https://pypi.org/project/heartpy/) is used to get a list of the peaks in the signal. The `hp.process()` function also further filters out very high and low heart rates, from 40 - 180 BPM.
Then we get the time difference interval between each peak (Inter-Beat Interval). Using the guidelines outlined by [(Camm et al., 1996)](doi.org/10.1093/oxfordjournals.eurheartj.a014868) we can then calculate the HRV in the form of Root Mean Square of Successive Differences (RMSSD).

In [5]:
def get_hrv_metrics(bvp_segment, timestamps, fs=400):
    if len(timestamps) != len(np.unique(timestamps)):
        print(f"Warning: The input 'timestamps' (bvp_ts_raw) array itself contains duplicate values. Number of timestamps: {len(timestamps)}, Unique timestamps: {len(np.unique(timestamps))}")
    if len(bvp_segment) < fs * 10:
        print(f"Warning: Segment too short ({len(bvp_segment)/fs:.2f}s, need at least 10s), skipping HRV.")
        return pd.Series(dtype=float)
    try:
        filtered_segment = filter_bvp(bvp_segment, fs=fs)
        wd, m = hp.process(filtered_segment, sample_rate=fs, calc_freq=False, high_precision=True, clean_rr=True)
        peaks = wd.get('peaklist', [])
        if len(peaks) < 5:
            print(f"Warning: Not enough peaks found ({len(peaks)}, need at least 5), skipping HRV.")
            return pd.Series(dtype=float)
        # Ensure peak indices are integers and within bounds before indexing timestamps
        valid_peaks = []
        for p in peaks:
            try:
                p_int = int(p) # Convert to integer
                if 0 <= p_int < len(timestamps): # Check bounds
                    valid_peaks.append(p_int)
            except (ValueError, TypeError):
                print(f"Warning: Invalid peak value {p} encountered, skipping it.")

        if len(valid_peaks) < 2:
            print(f"Warning: Not enough valid peaks ({len(valid_peaks)}) after filtering and conversion. Original peaks count from heartpy: {len(peaks)}. Skipping HRV.")
            return pd.Series(dtype=float)
        peak_times_sec = timestamps[valid_peaks]
        
        # debugging time differences
        time_diffs = np.diff(peak_times_sec)
        #print(f"Time differences between BVP samples: {time_diffs}")
        Trace(f"Mean time difference: {np.mean(time_diffs)}")
        Trace(f"Standard deviation of time differences: {np.std(time_diffs)}")
        
        ibi_ms = np.diff(peak_times_sec) * 1000
        if len(ibi_ms) < 3:
            print(f"Warning: Not enough IBIs calculated ({len(ibi_ms)}), skipping HRV.")
            return pd.Series(dtype=float)
        ibi_event_times_sec = peak_times_sec[1:]
        # debugging IBI to find dublicates
        Trace(f"unique ibi event times: {len(np.unique(ibi_event_times_sec))}")
        
        hrv_indices = nk.hrv({'RRI': ibi_ms, 'RRI_Time': ibi_event_times_sec}, sampling_rate=1000)
        return hrv_indices.iloc[0]
    except Exception as e:
        print(f"Error processing segment: {e}")
        return pd.Series(dtype=float)


### Running the functions

Below is the loop responsible for looping through the data files and grouping the different conditions. It ends by saving a data frame with each valid subject, the condition they tested, the phase, and HRV data.

In [6]:
def process_subject_data(subject_id, baseline_filepath, task_filepath, nominal_srate=400):
    results_list = []
    condition_from_task_folder = "Unknown"  # Default

    # --- Get Condition from Task File's PARENT FOLDER (if it exists) ---
    if task_filepath:
        task_parent_folder_name = os.path.basename(os.path.dirname(task_filepath))
        match_cond = re.search(r'_task-([^_\s]+)', task_parent_folder_name) # More robust regex for condition
        if match_cond:
            parsed_condition = match_cond.group(1)
            # Ensure 'Baseline' from folder name doesn't become the experimental condition
            if parsed_condition.lower() != 'baseline':
                condition_from_task_folder = parsed_condition
            else:
                print(f"Warning: Task filepath {task_filepath} seems to be from a folder named '...task-Baseline...'. Experimental condition remains 'Unknown' or will be based on a non-baseline task folder if available.")
        else:
            print(f"Warning: Could not parse condition from task folder name: {task_parent_folder_name} for subject {subject_id}")

    # --- 1. Process Baseline File ---
    if baseline_filepath:
        #print(f"Processing BASELINE for subject {subject_id} (Experimental Condition: {condition_from_task_folder}) from: {os.path.basename(baseline_filepath)}")
        bvp_baseline, ts_baseline, df_events_baseline, streams_baseline = extract_bvp_and_markers_from_xdf(baseline_filepath)

        if streams_baseline['bvp'] and bvp_baseline is not None and ts_baseline is not None:
            #print(f"Calculating baseline HRV for {subject_id} (full file)...")
            hrv_baseline_metrics = get_hrv_metrics(bvp_baseline, ts_baseline, fs=nominal_srate)
            if not hrv_baseline_metrics.empty:
                baseline_res = {'ParticipantID': subject_id, 'Condition': condition_from_task_folder, 'Phase': 'Baseline'}
                baseline_res.update(hrv_baseline_metrics)
                results_list.append(baseline_res)
                print(f"Baseline HRV calculated for {subject_id}.")
            else:
                print(f"No HRV metrics obtained for baseline for subject {subject_id}.")
        else:
            print(f"Could not process BVP for baseline for subject {subject_id} from {os.path.basename(baseline_filepath)}.")
    else:
        print(f"No baseline filepath provided for subject {subject_id}.")

    # --- 2. Process Task File ---
    if task_filepath:
        #print(f"Processing TASK for subject {subject_id}, Condition: {condition_from_task_folder} from: {os.path.basename(task_filepath)}")
        bvp_task_full, ts_task_full, df_events_task, streams_task = extract_bvp_and_markers_from_xdf(task_filepath)

        if streams_task['bvp'] and bvp_task_full is not None and ts_task_full is not None:
            #print(f"Calculating task HRV for {subject_id} (full file {len(bvp_task_full)/nominal_srate:.2f}s)...")
            hrv_task_metrics = get_hrv_metrics(bvp_task_full, ts_task_full, fs=nominal_srate)
            if not hrv_task_metrics.empty:
                task_res = {'ParticipantID': subject_id, 'Condition': condition_from_task_folder, 'Phase': 'Task'}
                task_res.update(hrv_task_metrics)
                results_list.append(task_res)
                print(f"Task HRV calculated for {subject_id}.")
            else:
                print(f"No HRV metrics obtained for task phase for subject {subject_id}.")
        else:
            print(f"Could not process BVP for task for subject {subject_id} from {os.path.basename(task_filepath)}.")
    else:
        print(f"No task filepath provided for subject {subject_id}.")

    return pd.DataFrame(results_list) if results_list else pd.DataFrame()

# --- Main Processing Loop ---
all_participant_dfs = [] 
data_root_folder = os.path.join(os.getcwd(), "Test_Data")
nominal_srate_main = 400

session_folders = [f.path for f in os.scandir(data_root_folder) if f.is_dir() and re.match(r'ses-\d+', f.name)]
Trace(f"Found session folders: {session_folders}")

for session_folder_path in session_folders:
    session_name = os.path.basename(session_folder_path)
    Trace(f"\nProcessing session: {session_name}")

    potential_st_folders = [f.path for f in os.scandir(session_folder_path) if f.is_dir() and "_task-" in f.name and "sub-" in f.name]

    subject_folders_map = {}
    for st_folder_path in potential_st_folders:
        st_folder_name = os.path.basename(st_folder_path)
        match_sub = re.search(r'(sub-[^_\s]+)', st_folder_name)
        if match_sub:
            subject_id_key = match_sub.group(1)
            if subject_id_key not in subject_folders_map:
                subject_folders_map[subject_id_key] = []
            subject_folders_map[subject_id_key].append(st_folder_path)
        else:
            print(f"  Warning: Could not parse subject ID from folder name {st_folder_name} in session {session_name}.")

    Trace(f"  Found data for subjects in {session_name}: {list(subject_folders_map.keys())}")

    for subject_id_key, folders_for_subject in subject_folders_map.items():
        Trace(f"    Processing data for subject key: {subject_id_key} in session: {session_name}")

        baseline_filepath = None
        task_condition_filepath = None

        for folder_path in folders_for_subject:
            folder_name = os.path.basename(folder_path)
            xdf_files_in_folder = glob.glob(os.path.join(folder_path, "*.xdf"))

            if not xdf_files_in_folder:
                print(f"      Warning: No XDF file found in folder {folder_name}. Skipping this folder.")
                continue
            if len(xdf_files_in_folder) > 1:
                print(f"      Warning: Multiple XDF files found in {folder_name}. Using the first one: {os.path.basename(xdf_files_in_folder[0])}.")
            current_xdf_file = xdf_files_in_folder[0]

            if "_task-Baseline" in folder_name:
                if baseline_filepath:
                    print(f"      Warning: Multiple baseline folders/files found for {subject_id_key} in {session_name}. Overwriting with data from {folder_name}.")
                baseline_filepath = current_xdf_file
                Trace(f"      Found Baseline file: {os.path.basename(baseline_filepath)} in folder {folder_name}")
            elif "_task-" in folder_name:
                if task_condition_filepath:
                    print(f"      Warning: Multiple task condition folders/files found for {subject_id_key} in {session_name}. Overwriting with data from {folder_name}.")
                task_condition_filepath = current_xdf_file
                #print(f"      Found Task Condition file: {os.path.basename(task_condition_filepath)} in folder {folder_name}")

        if baseline_filepath and task_condition_filepath:
            cleaned_subject_id = re.sub(r'^sub-', '', subject_id_key)
            Trace(f"      Pair found for {cleaned_subject_id}: Baseline ({os.path.basename(baseline_filepath)}), Task ({os.path.basename(task_condition_filepath)})")
            subject_hrv_df = process_subject_data(
                subject_id=cleaned_subject_id,
                baseline_filepath=baseline_filepath,
                task_filepath=task_condition_filepath,
                nominal_srate=nominal_srate_main
            )
            if not subject_hrv_df.empty:
                all_participant_dfs.append(subject_hrv_df)
            else:
                print(f"      No HRV data generated for subject {cleaned_subject_id} in session {session_name}.")
        else:
            missing_parts = []
            if not baseline_filepath: missing_parts.append("baseline file")
            if not task_condition_filepath: missing_parts.append("task condition file")
            print(f"      Skipping subject {subject_id_key} in session {session_name} due to missing { ' and '.join(missing_parts) }.")

if not all_participant_dfs:
    print("\nNo dataframes to concatenate. Final DataFrame will be empty.")
    final_df = pd.DataFrame()
else:
    final_df = pd.concat(all_participant_dfs, ignore_index=True)
    print("\n--- Combined Results DataFrame ---")
    if not final_df.empty:
        print(final_df.head())
    else:
        print("Final DataFrame is empty after processing all subjects.")

Baseline HRV calculated for 10.
Task HRV calculated for 10.
Could not process BVP for baseline for subject 11 from _.xdf.
Could not process BVP for task for subject 11 from _.xdf.
      No HRV data generated for subject 11 in session ses-11.
Could not process BVP for baseline for subject 12 from _.xdf.
Could not process BVP for task for subject 12 from _.xdf.
      No HRV data generated for subject 12 in session ses-12.
Could not process BVP for baseline for subject 13 from _.xdf.
Could not process BVP for task for subject 13 from _.xdf.
      No HRV data generated for subject 13 in session ses-13.
Baseline HRV calculated for 1.


/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/neurokit2/signal/signal_interpolate.py:117: NeuroKitWarning: Duplicate x values detected. Averaging their corresponding y values.
  warn(


Task HRV calculated for 1.


/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/neurokit2/signal/signal_interpolate.py:117: NeuroKitWarning: Duplicate x values detected. Averaging their corresponding y values.
  warn(


Baseline HRV calculated for 2.
Task HRV calculated for 2.


...

Task HRV calculated for 44.


/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/neurokit2/signal/signal_interpolate.py:117: NeuroKitWarning: Duplicate x values detected. Averaging their corresponding y values.
  warn(


Baseline HRV calculated for 45.


/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/neurokit2/signal/signal_interpolate.py:117: NeuroKitWarning: Duplicate x values detected. Averaging their corresponding y values.
  warn(


Task HRV calculated for 45.

--- Combined Results DataFrame ---
  ParticipantID Condition     Phase  HRV_MeanNN     HRV_SDNN  HRV_SDANN1  \
0            10  MediumFi  Baseline  660.764801   135.314503   31.175987   
1            10  MediumFi      Task  661.646606   815.397120  163.615276   
2             1    HighFi  Baseline  715.679284   226.257083   48.481917   
3             1    HighFi      Task  727.953191  1189.775585  273.490521   
4             2    HighFi  Baseline  565.038654    57.817954   16.013175   

   HRV_SDNNI1  HRV_SDANN2  HRV_SDNNI2  HRV_SDANN5  ...  HRV_SampEn  \
0  123.251099   17.586581  124.454787         NaN  ...    0.748622   
1  525.514743   87.186293  544.865326         NaN  ...    0.260511   
2  220.196165   40.496413  220.077430         NaN  ...    0.637484   
3  930.099612  110.209036  989.813782         NaN  ...    0.284032   
4   49.148874   19.703470   52.108503         NaN  ...    0.697417   

   HRV_ShanEn  HRV_FuzzyEn  HRV_MSEn  HRV_CMSEn  HRV_RCMSE

In [7]:
# Check a single subject's bvp stream to see if time stamps are evenly spaced
# Used for debugging
def check_bvp_timestamps(filepath):
    bvp_signal, ts, _, streams_found = extract_bvp_and_markers_from_xdf(filepath)
    if streams_found['bvp'] and bvp_signal is not None and ts is not None:
        print(f"BVP stream found in {filepath}.")
        time_diffs = np.diff(ts)
        print(f"Time differences between BVP samples: {time_diffs}")
        print(f"Mean time difference: {np.mean(time_diffs)}")
        print(f"Standard deviation of time differences: {np.std(time_diffs)}")
    else:
        print(f"No BVP stream found in {filepath}.")
check_bvp_timestamps("/home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/ses-27/sub-27_ses-27_task-MediumFi/_.xdf")

BVP stream found in /home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/ses-27/sub-27_ses-27_task-MediumFi/_.xdf.
Time differences between BVP samples: [0.0024997  0.0024997  0.0024997  ... 0.00249975 0.00249975 0.00249975]
Mean time difference: 0.002533121224245887
Standard deviation of time differences: 0.021291432581365042


## Statistical Model

With the DataFrame (`final_df`) containing HRV metrics for each participant, condition, and phase, we can compare the baseline vs task across the conditions.

A Linear Mixed-Effects Model (LMM) is used here. It allows us to model:
- **Fixed Effects:** The average effects of Phase (Baseline vs. Task) and Condition (LowFi, MedFi, HighFi), and their interaction (Phase * Condition).
- **Random Effects:** The variability between participants. They each have their own baseline level of the HRV metric unique to them, which is modeled as a random intercept (~1 ParticipantID).

In [8]:
import statsmodels.formula.api as smf

# Check if final_df exists and has data
if 'final_df' in locals() and not final_df.empty and 'HRV_RMSSD' in final_df.columns:
    # Ensure necessary columns are not all NaN
    if final_df[['HRV_RMSSD', 'Phase', 'Condition', 'ParticipantID']].isnull().all().any():
        print("Warning: One or more critical columns contain only NaN values. Cannot run model.")
    else: 
        # Remove rows with NaN in the outcome variable or predictors
        model_df = final_df.dropna(subset=['HRV_RMSSD', 'Phase', 'Condition', 'ParticipantID'])
        
        if model_df.empty:
            print("Warning: No valid data remaining after removing NaNs. Cannot run model.")
        else:
            #print("\n--- Inspecting model_df before fitting LMM ---")
            #print(model_df.info())

            # Print a few rows to see actual values
            print("\n--- Fitting Linear Mixed-Effects Model for HRV_RMSSD ---")
            # Ensure ParticipantID, Phase, and Condition are appropriate types
            model_df['ParticipantID'] = model_df['ParticipantID'].astype('category')
            model_df['Phase'] = model_df['Phase'].astype('category')
            model_df['Condition'] = model_df['Condition'].astype('category')
            
            # Define the model formula for fixed effects
            fixed_effects_formula = "HRV_RMSSD ~ C(Phase) * C(Condition)"
            # Define the random effects formula (random intercept for ParticipantID)
            random_effects_formula = "~1"
            
            try:
                # Fit the model using re_formula for random effects
                model = smf.mixedlm(fixed_effects_formula, 
                                  model_df, 
                                  groups=model_df["ParticipantID"], 
                                  re_formula=random_effects_formula)
                result = model.fit()
                
                # Print the summary
                print(result.summary())
            except Exception as e:
                print(f"Error fitting model: {e}")
                print("\nPlease check data structure and variability.")
                print("Model DataFrame head:")
                print(model_df.head())
else:
    print("Skipping statistical analysis: 'final_df' not created or is empty or missing 'HRV_RMSSD' column.")


--- Fitting Linear Mixed-Effects Model for HRV_RMSSD ---
                            Mixed Linear Model Regression Results
Model:                         MixedLM            Dependent Variable:            HRV_RMSSD   
No. Observations:              75                 Method:                        REML        
No. Groups:                    39                 Scale:                         1591039.6271
Min. group size:               1                  Log-Likelihood:                -601.9519   
Max. group size:               2                  Converged:                     Yes         
Mean group size:               1.9                                                           
---------------------------------------------------------------------------------------------
                                            Coef.    Std.Err.   z    P>|z|   [0.025   0.975] 
---------------------------------------------------------------------------------------------
Intercept                     

# Results

This mini-project aimed to create a pipeline for processing BVP data to extract HRV metrics; furthermore exploring the potential correlations with the user experience in a VR experience. The data was extracted using a Butterworth bandpass filter and utilized HeartPy and NeuoroKit2 to calculate the HRV RMSSD.

A LMM was used to analyze the data, considering the fixed effects of phase as baseline vs task, conditions as LowFi, MedFi, HighFi, and their interaction. While still accounting for random variability between participants.

The model showed indications that baseline RMSSD levels were comparable across the different conditions, which was expected. While not statistically significant at the p < 0.05 threshold, the interaction `C(Phase)[T.Task]:C(condition)[T.LowFi]` (HighFi reference group) saw a trend level of p = 0.074, where the LowFi condition might evoke a different physiological response. Specifically, the results hinted that the LowFi group experienced a smaller change in RMSSD during their task compared to the HighFi group.